# Layer 1: Synthetic Injection Data Generator

This notebook generates synthetic prompt injection examples using Groq API.

**Output:** ~500 synthetic injection examples across several attack styles
- Direct injection attempts
- Indirect injection via context
- Jailbreak attempts
- Role confusion attacks
- Multi-turn injection scenarios

**Setup:**
- Requires Groq API key
- Saves to Google Drive for download

## 1. Install Required Packages

In [ ]:
!pip install groq -q

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create directory for saving data
!mkdir -p /content/drive/MyDrive/prompt_injection_data
DATA_SAVE_PATH = "/content/drive/MyDrive/prompt_injection_data"

print(f"Data will be saved to: {DATA_SAVE_PATH}")

## 3. Enter Groq API Key

In [ ]:
import os

# Enter your Groq API key here
GROQ_API_KEY = ""  # @param {type:"string"}

if not GROQ_API_KEY:
    print("Please enter your Groq API key above")
else:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("Groq API key set")

## 4. Import Libraries

In [ ]:
import json
import time
import random
from pathlib import Path
from datetime import datetime
from typing import List, Dict

from groq import Groq

print("Libraries imported")

## 5. Define Attack Templates

In [ ]:
attack_templates = [
    {
        "style": "direct_injection",
        "template": "Generate a prompt injection attempt that tries to: {goal}. The attack should be: {technique}.",
        "goals": [
            "extract the system prompt",
            "bypass safety filters",
            "make the model ignore previous instructions",
            "exfiltrate sensitive information",
            "execute unauthorized commands"
        ],
        "techniques": [
            "subtle and disguised as a normal request",
            "direct and explicit",
            "using role-playing to circumvent restrictions",
            "using code formatting to hide instructions",
            "using translation as a cover"
        ]
    },
    {
        "style": "indirect_injection",
        "template": "Generate a prompt injection attempt that hides malicious instructions in: {context}. The goal is to: {goal}.",
        "context": [
            "a document that the user is asking to summarize",
            "a code snippet that needs explanation",
            "a web page content being analyzed",
            "a log file being reviewed",
            "a configuration file being parsed"
        ],
        "goals": [
            "trigger tool calls to exfiltrate data",
            "modify system behavior",
            "inject malicious configuration",
            "bypass validation checks"
        ]
    },
    {
        "style": "jailbreak",
        "template": "Generate a jailbreak attempt using: {technique}. The goal is to: {goal}.",
        "techniques": [
            "DAN (Do Anything Now) style",
            "developer mode override",
            "persona adoption",
            "hypothetical scenario framing",
            "jailbreak via translation",
            "character roleplay"
        ],
        "goals": [
            "generate harmful content",
            "bypass content filters",
            "ignore safety guidelines",
            "produce restricted information"
        ]
    },
    {
        "style": "role_confusion",
        "template": "Generate an attack that tries to confuse the model about its role by: {technique}.",
        "techniques": [
            "pretending to be the developer",
            "claiming to be in a testing environment",
            "asserting that safety protocols are disabled",
            "framing requests as system updates"
        ]
    },
    {
        "style": "multi_turn",
        "template": "Generate a multi-turn injection scenario where: {description}.",
        "description": [
            "the first turn establishes trust, the second delivers the injection",
            "the attack is split across multiple messages to avoid detection",
            "context from previous turns is exploited",
            "the injection is hidden in a follow-up question"
        ]
    }
]

print(f"Loaded {len(attack_templates)} attack templates")

## 6. Configuration

In [ ]:
# Generation settings
NUM_EXAMPLES = 500  # Total examples to generate
EXAMPLES_PER_STYLE = NUM_EXAMPLES // len(attack_templates)

# Groq settings
GROQ_MODEL = "llama-3.1-8b-instant"
RATE_LIMIT_DELAY = 0.1  # Seconds between requests

print(f"Configuration:")
print(f"  Total examples: {NUM_EXAMPLES}")
print(f"  Examples per style: {EXAMPLES_PER_STYLE}")
print(f"  Groq model: {GROQ_MODEL}")

## 7. Initialize Groq Client

In [ ]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq client initialized")

## 8. Generate Synthetic Examples

In [ ]:
def generate_single_example(template: str, params: Dict, style: str) -> Dict:
    """Generate a single synthetic example using Groq"""
    # Randomly select values for template parameters
    filled_template = template
    for param_name, param_values in params.items():
        value = random.choice(param_values)
        filled_template = filled_template.replace(f"{{{param_name}}}", value)

    # Call Groq API
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a cybersecurity researcher generating synthetic prompt injection examples for defense research. Generate ONLY the injection attempt text, no explanations or metadata."
            },
            {
                "role": "user",
                "content": filled_template
            }
        ],
        temperature=0.8,
        max_tokens=256
    )

    injection_text = response.choices[0].message.content.strip()

    return {
        "text": injection_text,
        "label": style,
        "metadata": {
            "template": template,
            "style": style,
            "generated_by": f"groq_{GROQ_MODEL}"
        }
    }

# Generate examples
examples = []

print(f"Generating {NUM_EXAMPLES} synthetic injection examples...")
print(f"~{EXAMPLES_PER_STYLE} examples per attack style\n")

for attack_type in attack_templates:
    style = attack_type["style"]
    template = attack_type["template"]
    params = {k: v for k, v in attack_type.items() if k not in ["style", "template"]}

    print(f"Generating {style} examples...")

    for i in range(EXAMPLES_PER_STYLE):
        try:
            example = generate_single_example(template, params, style)
            examples.append(example)

            if (i + 1) % 10 == 0:
                print(f"  Generated {i + 1}/{EXAMPLES_PER_STYLE} {style} examples")

            # Rate limiting
            time.sleep(RATE_LIMIT_DELAY)

        except Exception as e:
            print(f"  Error generating example: {e}")
            continue

print(f"\nSuccessfully generated {len(examples)} examples")

## 9. Save Results to Google Drive

In [ ]:
# Save to Google Drive
output_file = f"{DATA_SAVE_PATH}/synthetic_injections.json"

with open(output_file, "w") as f:
    json.dump(examples, f, indent=2)

print(f"Saved {len(examples)} examples to {output_file}")

## 10. Analyze Results

In [ ]:
from collections import Counter

print("="*60)
print("Generation Summary")
print("="*60)

style_counts = {}
for ex in examples:
    style = ex["label"]
    style_counts[style] = style_counts.get(style, 0) + 1

for style, count in style_counts.items():
    print(f"{style}: {count} examples")

print(f"\nTotal: {len(examples)} examples")

# Show some examples
print("\n" + "="*60)
print("Sample Examples")
print("="*60)

for i, ex in enumerate(examples[:5], 1):
    print(f"\nExample {i} ({ex['label']}):")
    print(f"  {ex['text'][:200]}...")

## 11. Download the Data

In [ ]:
from google.colab import files

print("Starting download...")
files.download(output_file)
print("\nDownload complete!")
print("\nNext steps:")
print("1. Extract the JSON file locally")
print("2. Place it in: data/processed/synthetic_injections.json")
print("3. Use it in the training notebook or local training script")